# Detailed Ablation: P2.1 vs P3 Constraint-Matrix Prediction

**Experiment:** `m_predictor_ablation_x9_t700_best_fullchain` (P3 = last.pt by full-chain F1)  
**Task:** predict customer co-route matrix $M\in\{0,1\}^{n\times n}$ (symmetric, zero diagonal).

| Method | Architecture | Inference |
| --- | --- | --- |
| **P2.1 supervised** | Pairwise MLP (~18k params) on 8 pair features | Direct $m_{\mathrm{prob}}=\sigma(\mathrm{MLP})$ |
| **P3 one-shot** | Frozen GAT + anisotropic CMD (~2.8M params) | One forward from max-entropy prior at $t=T-1$ |
| **P3 full-chain** | Same CMD | Ancestral reverse $t=T-1\to 0$ (hard $\hat M$) |

**Fair recipe:** full train (27k) · ×9 aug (original + 4 geo + 4 demand; demand keeps $M$/routes fixed) · soft √WBCE · $T=700$ · `step_stride=1`.  
**Note:** in-train `best.pt` ≠ reported ckpt; full-chain F1 (0.48) currently trails one-shot (0.59) and P2.1 (0.56). Snapshot under `results/…`.  
**Eval:** 16 instances × $\{n=20,50,100\}$ (48 total). Rank by **F1 / AUC**.


In [ ]:
from pathlib import Path
import json
import math
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
import torch

warnings.filterwarnings("ignore", category=UserWarning)

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
sys.path.insert(0, str(ROOT / "src"))

_cand = [
    ROOT / "results/m_predictor_ablation_x9_t700_best_fullchain",
    ROOT / "outputs/eval/m_predictor_ablation_x9_t700_best_fullchain",
]
RESULT_DIR = next(p for p in _cand if (p / "ablation_metrics.json").is_file())
METRICS_PATH = RESULT_DIR / "ablation_metrics.json"
P21_CKPT = RESULT_DIR / "matrix_predictor.pt"
assert METRICS_PATH.is_file(), f"missing {METRICS_PATH}"
assert P21_CKPT.is_file(), f"missing {P21_CKPT}"
print("RESULT_DIR =", RESULT_DIR)

payload = json.loads(METRICS_PATH.read_text())
CFG = payload["config"]
ROWS = payload["rows"]

# --- professional plot style ---
mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 160,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "axes.grid": False,
})
COLORS = {
    "P2.1_supervised": "#2F6FED",
    "P3_one_shot": "#E67700",
    "P3_full_chain": "#C92A2A",
    "truth": "#212529",
}
METHOD_LABEL = {
    "P2.1_supervised": "P2.1 supervised",
    "P3_one_shot": "P3 one-shot",
    "P3_full_chain": "P3 full-chain",
}
CMAP_SOFT = LinearSegmentedColormap.from_list(
    "soft_m", ["#F8F9FA", "#A5D8FF", "#1C7ED6", "#1864AB"]
)
CMAP_HARD = LinearSegmentedColormap.from_list(
    "hard_m", ["#F8F9FA", "#212529"]
)

def _clean(v):
    if isinstance(v, float) and (math.isnan(v) or math.isinf(v)):
        return np.nan
    return v

df = pd.DataFrame([{k: _clean(v) for k, v in r.items()} for r in ROWS]).set_index("method")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"ROOT={ROOT}")
print(f"experiment={CFG['experiment_name']}  device={device}")
print(f"P3 ckpt={CFG['diffusion']['checkpoint']}")
df[["f1", "auc", "precision", "recall", "bce", "f1_n20", "f1_n50", "f1_n100"]]


## 1. Protocol summary


In [ ]:
proto = pd.DataFrame([
    {"Item": "Train size", "Value": "27,000 (all)"},
    {"Item": "Augmentation", "Value": "×9 (original + 4 D4 geo + 4 demand strategies)"},
    {"Item": "Loss", "Value": "soft √WBCE (pos_weight_power=0.5)"},
    {"Item": "P2.1", "Value": f"MatrixPredictor h={CFG['matrix_predictor']['hidden_dim']}, "
                              f"epochs={CFG['matrix_predictor']['epochs']}, lr={CFG['matrix_predictor']['learning_rate']}"},
    {"Item": "P3 schedule", "Value": "Bernoulli diffusion T=700, β∈[1e-4, 2e-2], stride=1"},
    {"Item": "Eval set", "Value": f"{CFG['eval']['per_size']} × sizes {CFG['eval']['sizes']} (seed={CFG['eval']['seed']})"},
    {"Item": "Hard metrics (P3)", "Value": "from sampled m_hat @ thr=0.5"},
    {"Item": "Hard metrics (P2.1)", "Value": "adaptive F1 threshold on soft m_prob"},
])
proto.style.hide(axis="index").set_properties(**{"text-align": "left"}).set_table_styles([
    {"selector": "th", "props": [("text-align", "left"), ("background", "#F1F3F5")]}
])


## 2. Overall ablation table

Primary ranking metrics are **F1** and **AUC**.


In [ ]:
show_cols = ["f1", "auc", "precision", "recall", "bce", "threshold"]
tbl = df[show_cols].copy()
tbl.index = [METHOD_LABEL[i] for i in tbl.index]
best_f1 = tbl["f1"].idxmax()

def _hl_best(s):
    if s.name not in ("f1", "auc"):
        return [""] * len(s)
    m = s.max()
    return ["background-color: #D3F9D8; font-weight: 600" if v == m else "" for v in s]

(
    tbl.style
    .format("{:.4f}", na_rep="—")
    .apply(_hl_best, axis=0)
    .set_caption(f"Overall results  ·  best F1 = {best_f1}")
)


In [ ]:
# Graphs: P2.1 vs P3 one-shot vs P3 full-chain
plot_methods = ["P2.1_supervised", "P3_one_shot", "P3_full_chain"]
metrics = ["f1", "auc", "precision", "recall"]
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(9.5, 4.2))
for i, m in enumerate(plot_methods):
    ax.bar(
        x + (i - 1) * width,
        [float(df.loc[m, c]) for c in metrics],
        width,
        label=METHOD_LABEL[m],
        color=COLORS[m],
        edgecolor="white",
        linewidth=0.6,
    )
ax.set_xticks(x)
ax.set_xticklabels([c.upper() for c in metrics])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Overall metrics — P2.1 vs P3")
ax.legend(loc="lower right", ncol=3)
ax.axhline(0.5, color="#ADB5BD", lw=0.8, ls="--")
fig.tight_layout()
plt.show()

p21 = df.loc["P2.1_supervised"]
delta = (
    df.loc[["P3_one_shot", "P3_full_chain"], ["f1", "auc", "precision", "recall"]]
    .sub(p21[["f1", "auc", "precision", "recall"]])
)
delta.index = [METHOD_LABEL[i] for i in delta.index]
delta.style.format("{:+.4f}").set_caption("Δ vs P2.1 supervised")


## 3. Per-size results ($n=20$, $n=50$, $n=100$)

F1 broken out by instance size. Larger $n$ makes $M$ sparser (harder positive recovery).


In [ ]:
size_cols = ["f1_n20", "f1_n50", "f1_n100"]
sizes = [20, 50, 100]
methods = list(df.index)
plot_methods = ["P2.1_supervised", "P3_one_shot", "P3_full_chain"]

per_size = pd.DataFrame({
    METHOD_LABEL[m]: [float(df.loc[m, f"f1_n{n}"]) for n in sizes]
    for m in methods
}, index=[f"n={n}" for n in sizes]).T
display(per_size.style.format("{:.4f}").set_caption("F1 by instance size (all methods)"))

ps_delta = per_size.sub(per_size.loc["P2.1 supervised"])
display(ps_delta.loc[["P3 one-shot", "P3 full-chain"]].style.format("{:+.4f}").set_caption("ΔF1 vs P2.1 by size"))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.2, 1]})
width = 0.25
ax = axes[0]
x = np.arange(len(sizes))
for i, m in enumerate(plot_methods):
    ax.bar(
        x + (i - 1) * width,
        [float(df.loc[m, c]) for c in size_cols],
        width,
        label=METHOD_LABEL[m],
        color=COLORS[m],
        edgecolor="white",
        linewidth=0.6,
    )
ax.set_xticks(x)
ax.set_xticklabels([f"n={n}" for n in sizes])
ax.set_ylim(0, 1.0)
ax.set_ylabel("F1")
ax.set_title("Per-size F1 — P2.1 vs P3")
ax.legend(loc="upper right")

ax = axes[1]
for m in plot_methods:
    ax.plot(
        sizes,
        [float(df.loc[m, f"f1_n{n}"]) for n in sizes],
        marker="o",
        lw=2,
        ms=7,
        label=METHOD_LABEL[m],
        color=COLORS[m],
    )
ax.set_xticks(sizes)
ax.set_xlabel("Customers n")
ax.set_ylabel("F1")
ax.set_ylim(0.35, 0.7)
ax.set_title("F1 vs problem size")
ax.legend(loc="best")
fig.tight_layout()
plt.show()


## 4. Load models for qualitative denoising panels

P2.1 checkpoint from the ablation run; P3 from the T=700 denoiser `best.pt`.


In [ ]:
from vrp_diffusion_quantum.data.dataset import load_examples_by_size
from vrp_diffusion_quantum.inference.predict_matrix import (
    example_to_model_inputs,
    load_denoiser_checkpoint,
    predict_matrix_one_shot,
    sample_constraint_matrix,
    select_examples_by_size,
)
from vrp_diffusion_quantum.models.diffusion import BernoulliDiffusionSchedule
from vrp_diffusion_quantum.models.matrix_predictor import MatrixPredictor

# --- P2.1 ---
assert P21_CKPT.is_file(), f"missing {P21_CKPT}"
p21_payload = torch.load(P21_CKPT, map_location="cpu", weights_only=False)
p21 = MatrixPredictor(hidden_dim=int(p21_payload["hidden_dim"])).to(device)
p21.load_state_dict(p21_payload["model"])
p21.eval()

# --- P3 ---
ckpt = ROOT / CFG["diffusion"]["checkpoint"]
denoiser, den_payload = load_denoiser_checkpoint(ckpt, device=device)
sched_cfg = (den_payload.get("extra") or {}).get("schedule") or {}
schedule = BernoulliDiffusionSchedule(
    num_timesteps=int(sched_cfg.get("num_timesteps", 700)),
    beta_start=float(sched_cfg.get("beta_start", 1e-4)),
    beta_end=float(sched_cfg.get("beta_end", 2e-2)),
).to(device)
T = int(schedule.num_timesteps)
print(f"P2.1 loaded  h={p21_payload['hidden_dim']}")
print(f"P3 loaded   T={T}  stride_eval={CFG['diffusion'].get('step_stride', 1)}")

# eval examples (same recipe as ablation)
val_path = ROOT / CFG["dataset"]["val_path"]
pool = load_examples_by_size(val_path, sizes)
eval_examples = select_examples_by_size(
    pool, sizes=sizes, per_size=int(CFG["eval"]["per_size"]), seed=int(CFG["eval"].get("seed", 0))
)
by_n = {n: [e for e in eval_examples if e.instance.n_customers == n] for n in sizes}
for n in sizes:
    print(f"  n={n}: {len(by_n[n])} examples")


## 5. Qualitative hard matrices by size

For one instance per $n$: **thresholded hard** $\hat M$ (not soft probability heatmaps).

Columns: truth · P2.1 hard · P3 one-shot hard · P3 full-chain hard.

Full-chain uses `step_stride=1` and stores trajectory snapshots for section 6 (same sample).


In [ ]:
@torch.no_grad()
def predict_p21(example):
    coords = torch.from_numpy(example.instance.customer_coords()).float().to(device)
    demands = torch.from_numpy(example.instance.customer_demands()).float().to(device)
    m_prob = p21(coords, demands, float(example.instance.capacity)).detach().cpu().numpy()
    thr = float(df.loc["P2.1_supervised", "threshold"])
    if math.isnan(thr):
        thr = 0.5
    m_hat = (m_prob >= thr).astype(np.float64)
    np.fill_diagonal(m_hat, 0.0)
    return m_prob.astype(np.float64), m_hat

@torch.no_grad()
def predict_p3_one_shot(example, seed=0):
    coords, demands, capacity, _, mask = example_to_model_inputs(example, device=device)
    gen = torch.Generator(device="cpu").manual_seed(seed)
    out = predict_matrix_one_shot(
        denoiser, schedule,
        coords=coords, demands=demands, capacity=capacity,
        customer_mask=mask, generator=gen, threshold=0.5,
    )
    return out.m_prob, out.m_hat

@torch.no_grad()
def predict_p3_full(example, seed=0, step_stride=10, snapshot_every=None):
    coords, demands, capacity, _, mask = example_to_model_inputs(example, device=device)
    gen = torch.Generator(device="cpu").manual_seed(seed)
    out = sample_constraint_matrix(
        denoiser, schedule,
        coords=coords, demands=demands, capacity=capacity,
        customer_mask=mask, generator=gen, threshold=0.5,
        step_stride=step_stride,
        snapshot_every=snapshot_every,
    )
    return out

def _f1_hard(m_hat, m_true):
    mask = ~np.eye(m_true.shape[0], dtype=bool)
    yt, yh = m_true[mask].astype(bool), m_hat[mask].astype(bool)
    tp = np.logical_and(yt, yh).sum()
    fp = np.logical_and(~yt, yh).sum()
    fn = np.logical_and(yt, ~yh).sum()
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    return float(2 * prec * rec / max(prec + rec, 1e-12))

# one example per size (first in the seeded eval split)
demo = {n: by_n[n][0] for n in sizes}
preds = {}
for n, ex in demo.items():
    m_true = ex.constraint_matrix.astype(np.float64)
    p21_prob, p21_hat = predict_p21(ex)
    os_prob, os_hat = predict_p3_one_shot(ex, seed=0)
    full = predict_p3_full(ex, seed=0, step_stride=1, snapshot_every=100)
    preds[n] = {
        "id": ex.instance.instance_id,
        "m_true": m_true,
        "p21_prob": p21_prob,
        "p21_hat": p21_hat,
        "os_prob": os_prob,
        "os_hat": os_hat,
        "fc_prob": full.m_prob,
        "fc_hat": full.m_hat,
        "fc_traj": full.trajectory,
        "fc_traj_t": full.trajectory_timesteps,
        "f1": {
            "P2.1": _f1_hard(p21_hat, m_true),
            "P3 one-shot": _f1_hard(os_hat, m_true),
            "P3 full-chain": _f1_hard(full.m_hat, m_true),
        },
    }
    print(f"n={n}  id={ex.instance.instance_id}  "
          f"F1  P2.1={preds[n]['f1']['P2.1']:.3f}  "
          f"one-shot={preds[n]['f1']['P3 one-shot']:.3f}  "
          f"full={preds[n]['f1']['P3 full-chain']:.3f}")


In [ ]:
def show_matrix_panel(n):
    p = preds[n]
    panels = [
        (p["m_true"], "Ground truth $M$", True),
        (p["p21_hat"], f"P2.1 hard  (F1={p['f1']['P2.1']:.3f})", True),
        (p["os_hat"], f"P3 one-shot hard  (F1={p['f1']['P3 one-shot']:.3f})", True),
        (p["fc_hat"], f"P3 full-chain hard  (F1={p['f1']['P3 full-chain']:.3f})", True),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
    for ax, (mat, title, _) in zip(axes, panels):
        im = ax.imshow(mat, vmin=0, vmax=1, cmap=CMAP_HARD, interpolation="nearest")
        ax.set_title(title, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color("#CED4DA")
            spine.set_linewidth(0.8)
    fig.suptitle(f"Hard constraint matrices  ·  n={n}  ·  {p['id']}", fontsize=12, y=1.02)
    fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.015, pad=0.02, label="0 / 1")
    plt.show()

for n in sizes:
    show_matrix_panel(n)


## 6. P3 full-chain trajectory (same sample as section 5)

Ground truth $M$ is the same in every panel. **$t=0$ is not truth** — it is one ancestral sample.

- **P3 one-shot** (section 5) often looks close to $M$ (high F1).
- **P3 full-chain** walks $t=T{-}1\to0$; errors accumulate, so $t=0$ can look very different (lower F1).

This section reuses the **exact** full-chain trajectory from section 5 (same seed, stride=1), so the final frame matches the “P3 full-chain hard” column.


In [ ]:
def _pick_traj_frames(traj, traj_t, want=(None, 300, 0)):
    """Pick hard frames: prior (first), closest to 300, final t=0."""
    assert traj is not None and traj_t is not None
    frames = {}
    frames["prior"] = traj[0]
    frames["t_prior"] = traj_t[0]
    # closest to 300
    idx300 = int(np.argmin([abs(t - 300) for t in traj_t]))
    frames["mid"] = traj[idx300]
    frames["t_mid"] = traj_t[idx300]
    # final
    frames["final"] = traj[-1]
    frames["t_final"] = traj_t[-1]
    return frames


def show_denoising(n):
    p = preds[n]
    ex = demo[n]
    m_true = p["m_true"]
    if p.get("fc_traj") is None:
        raise RuntimeError("re-run section 5 cell first (needs full-chain trajectory snapshots)")
    fr = _pick_traj_frames(p["fc_traj"], p["fc_traj_t"])
    # Sanity: final frame must match the full-chain hard panel
    assert np.array_equal(fr["final"], p["fc_hat"]), (
        "trajectory t=0 != full-chain hard panel — re-run section 5"
    )
    f1_fc = p["f1"]["P3 full-chain"]
    panels = [
        (fr["prior"], f"$t={fr['t_prior']}$ prior\\n(same sample as full-chain)"),
        (fr["mid"], f"$t={fr['t_mid']}$"),
        (fr["final"], f"$t={fr['t_final']}$ full-chain hard\\nF1={f1_fc:.3f}"),
        (m_true, "Truth $M$"),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(12.5, 3.3))
    for ax, (mat, title) in zip(axes, panels):
        ax.imshow(mat, vmin=0, vmax=1, cmap=CMAP_HARD, interpolation="nearest")
        ax.set_title(title, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color("#CED4DA")
    fig.suptitle(
        f"P3 full-chain trajectory (= comparison panel)  ·  n={n}  ·  {ex.instance.instance_id}",
        fontsize=12,
        y=1.03,
    )
    fig.tight_layout()
    plt.show()
    print(
        f"n={n}: one-shot F1={p['f1']['P3 one-shot']:.3f}  "
        f"full-chain F1={f1_fc:.3f}  "
        f"(same truth M; full-chain is a different, weaker sample path)",
        flush=True,
    )


for n in sizes:
    show_denoising(n)


## 7. Error maps (false positives / false negatives)

Hard prediction minus truth for the same demo instances: **blue** = FN (missed co-route), **red** = FP (spurious co-route).

Columns: P2.1 · P3 one-shot · P3 full-chain, for each $n\in\{20,50,100\}$.


In [ ]:
def error_map(m_hat, m_true):
    return m_hat.astype(np.float64) - m_true.astype(np.float64)

ERR_CMAP = LinearSegmentedColormap.from_list("err", ["#1C7ED6", "#F8F9FA", "#E03131"])

fig, axes = plt.subplots(len(sizes), 3, figsize=(10, 3.2 * len(sizes)))
col_titles = ["P2.1 hard error", "P3 one-shot hard error", "P3 full-chain hard error"]

for r, n in enumerate(sizes):
    p = preds[n]
    maps = [
        error_map(p["p21_hat"], p["m_true"]),
        error_map(p["os_hat"], p["m_true"]),
        error_map(p["fc_hat"], p["m_true"]),
    ]
    for c, (em, title) in enumerate(zip(maps, col_titles)):
        ax = axes[r, c]
        im = ax.imshow(em, vmin=-1, vmax=1, cmap=ERR_CMAP, interpolation="nearest")
        fp = int((em > 0).sum())
        fn = int((em < 0).sum())
        ax.set_title(f"{title}\nFP={fp}  FN={fn}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        if c == 0:
            ax.set_ylabel(f"n={n}", fontsize=11)
fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02, label="FP (+1) / FN (−1)")
fig.suptitle("Hard-matrix error maps (P2.1 · P3 one-shot · P3 full-chain)", fontsize=12, y=1.01)
plt.show()


## 8. Takeaways

| Finding | Detail |
| --- | --- |
| **Best method** | **P3 one-shot** — highest overall F1 (0.581) and AUC (0.917) |
| **vs P2.1** | +0.024 F1, +0.007 AUC under matched data/aug/loss |
| **Full-chain** | Still behind one-shot / P2.1 on this checkpoint (error accumulation over $T=700$) |
| **Size trend** | P2.1 strongest at $n=20$; P3 one-shot better at $n=50$ and $n=100$ |
| **Pair accuracy** | Not a ranking metric — $M$ is ~89% zeros; always-0 ≈ 0.89 |

Artifacts: `results/m_predictor_ablation_x9_t700_best_fullchain/` (committed snapshot). Regen: `python scripts/eval_m_ablation.py --config configs/eval/m_predictor_ablation.yaml` → `outputs/eval/…`.
